# LimeSurvey Analysis Notebook

This notebook loads and explores LimeSurvey exports.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')

In [2]:
# Update filename if needed
csv_path = 'results-survey432293.csv'
df = pd.read_csv(csv_path)

print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 15, Columns: 135


,id,submitdate,lastpage,startlanguage,seed,startdate,datestamp,Q00001[SQ001],Q00001[SQ002],Q00001[SQ003],...,groupTime5717,Q00030Time,Q00031Time,Q00032Time,Q00033Time,Q00034Time,groupTime5708,Q00035Time,Q00036Time,Q00037Time
0,2,2026-03-25 11:03:14,9,en,1851706079,2026-03-25 10:44:51,2026-03-25 11:03:14,No,Yes,No,...,20.58,NaN,NaN,NaN,NaN,NaN,5.30,NaN,NaN,NaN
1,4,2026-03-25 11:06:59,9,en,663332136,2026-03-25 10:45:13,2026-03-25 11:06:59,No,Yes,No,...,36.65,NaN,NaN,NaN,NaN,NaN,64.01,NaN,NaN,NaN
2,6,2026-03-25 11:04:24,9,en,386207750,2026-03-25 10:45:17,2026-03-25 11:04:24,No,Yes,No,...,35.19,NaN,NaN,NaN,NaN,NaN,3.16,NaN,NaN,NaN
3,7,2026-03-25 11:02:06,9,en,1234118210,2026-03-25 10:45:43,2026-03-25 11:02:06,No,No,No,...,15.19,NaN,NaN,NaN,NaN,NaN,56.62,NaN,NaN,NaN
4,8,2026-03-25 11:05:12,9,en,1444895015,2026-03-25 10:46:41,2026-03-25 11:05:12,No,Yes,No,...,26.27,NaN,NaN,NaN,NaN,NaN,72.29,NaN,NaN,NaN


# Cleanup unused columns

In [3]:
# change the index to the id column

df = df.set_index("id")

In [4]:
# drop all columns starting with Q00007..Q00011
# (intro questions + object comprehension) no need for that

prefixes = [f"Q{i:05d}" for i in range(7, 12)]
#text field
prefixes.append("Q00026")
prefixes.append("Q00014")
prefixes.append("Q00030")


cols_to_drop = [
    col for col in df.columns
    if any(col.startswith(prefix) for prefix in prefixes)
]
print(cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")

['Q00007', 'Q00008', 'Q00009', 'Q00010', 'Q00011[SQ001]', 'Q00011[SQ002]', 'Q00011[SQ003]', 'Q00011[SQ004]', 'Q00011[SQ005]', 'Q00014', 'Q00026', 'Q00030', 'Q00007Time', 'Q00008Time', 'Q00009Time', 'Q00010Time', 'Q00011Time', 'Q00014Time', 'Q00026Time', 'Q00030Time']
Dropped 20 columns


# Creating Subsets


Expertise Level Description
Aviation Domain Expert Active or former Air Traffic Controller
Aviation Semi-Domain Expert Person working or researching on ATM in general
Aviation Layperson Any other person without sufficient knowledge about
ATM or ATC

Expertise Level Description
AI Domain Expert Studied or researches AI-relevant subjects and has
knowledge about XAI and RL
AI Semi-Domain Expert Person having an understanding of AI but not about
XAI
AI Layperson Any other person without sufficient knowledge about


In [5]:

atc_knowledge_levels = [
    "Working Knowledge: I understand ATC procedures and operational context in depth (e.g., work in ATM, studied it formally)",
    "Expert: I have direct operational experience as a controller or deep professional ATC expertise",
    "Conceptual Understanding: I understand key ATC concepts, terminology, and challenges without operational experience",
]

mask_domain = df["Q00001[SQ001]"].eq("Yes")  # active/former ATC
mask_semibase = df["Q00001[SQ002]"].eq("Yes")  # ATM-related work/research
mask_knowledge = df["Q00002"].isin(atc_knowledge_levels)

# Make groups disjoint:
mask_semi = (~mask_domain) & mask_semibase & mask_knowledge
mask_layman = ~(mask_domain | mask_semi)

AviationDomainExperts = df[mask_domain]
AviationSemiDomainExperts = df[mask_semi]
AviationLayman = df[mask_layman]

# Optional check:
print(len(AviationDomainExperts), len(AviationSemiDomainExperts), len(AviationLayman), len(df))


0 4 11 15


In [6]:


ml_expert_knowledge_levels = [
    "Technical: I understand core mechanics (e.g., gradient descent, loss functions, model architectures) and can work with them confidently",
    "Expert: I have deep theoretical and practical ML knowledge, including mathematical foundations, and can critically analyze or develop models"
]
ml_semi_knowledge_levels = [
    "Applied: I can use ML tools/libraries without fully understanding the math",
    "Technical: I understand core mechanics (e.g., gradient descent, loss functions, model architectures) and can work with them confidently",
    "Expert: I have deep theoretical and practical ML knowledge, including mathematical foundations, and can critically analyze or develop models"
]

mask_ai_domain_experts = (df["Q00001[SQ004]"].eq("Yes") | df["Q00001[SQ003]"].eq("Yes"))  # Student or aiml reaearch professional
mask_rl_xrl = (df["Q00004"].eq("Yes") & df["Q00005"].eq("Yes")) # knowledge a bout XRL RL
mask_not_xai = df["Q00005"].eq("No")

mask_expert_knowledge = df["Q00003"].isin(ml_expert_knowledge_levels)
mask_semi_knowledge = df["Q00003"].isin(ml_semi_knowledge_levels)

mask_expert = (mask_ai_domain_experts & mask_rl_xrl & mask_expert_knowledge)
mask_semi = (~mask_expert & mask_semi_knowledge)
mask_layman = ~(mask_expert | mask_semi)

AiDomainExperts = df[mask_expert]
AiSemiDomainExperts = df[mask_semi]
AiLayman= df[mask_layman]

print(len(AiDomainExperts), len(AiSemiDomainExperts), len(AiLayman), len(df))


2 8 5 15


In [7]:
groups = {
    "All Participants": df,
    "AI Domain Experts": AiDomainExperts,
    "AI Semi-Domain Experts": AiSemiDomainExperts,
    "AI Layman": AiLayman,
    "Aviation Domain Experts": AviationDomainExperts,
    "Aviation Semi-Domain Experts": AviationSemiDomainExperts,
    "Aviation Layman": AviationLayman,
}


In [8]:
# Question-code pairs for pre/post comparison

mental_model_pairs = {
    "saliency_safe_state": {"pre": "Q00012", "post": "Q00022"},
    "saliency_conflict_state": {"pre": "Q00013", "post": "Q00021"},
    "moe": {"pre": "Q00015", "post": "Q00024"},
    "action_heatmaps": {"pre": "Q00016", "post": "Q00025"},
}

explanation_trust_pairs = {
    "trust_item_1": {"pre": "Q00017", "post": "Q00031"},
    "trust_item_2": {"pre": "Q00018", "post": "Q00032"},
    "trust_item_3": {"pre": "Q00019", "post": "Q00033"},
    "trust_item_4": {"pre": "Q00020", "post": "Q00034"},
}

question_pairs = {
    "mental_models": mental_model_pairs,
    "explanation_trust": explanation_trust_pairs,
}

likert_scale = {"I agree strongly": 2,
                "I agree somewhat": 1,
                "I’m neutral about it": 0,
                "I disagree somewhat": -1,
                "I disagree strongly": -2
                
                }

question_pairs

{'mental_models': {'saliency_safe_state': {'pre': 'Q00012', 'post': 'Q00022'},
  'saliency_conflict_state': {'pre': 'Q00013', 'post': 'Q00021'},
  'moe': {'pre': 'Q00015', 'post': 'Q00024'},
  'action_heatmaps': {'pre': 'Q00016', 'post': 'Q00025'}},
 'explanation_trust': {'trust_item_1': {'pre': 'Q00017', 'post': 'Q00031'},
  'trust_item_2': {'pre': 'Q00018', 'post': 'Q00032'},
  'trust_item_3': {'pre': 'Q00019', 'post': 'Q00033'},
  'trust_item_4': {'pre': 'Q00020', 'post': 'Q00034'}}}

In [9]:
def calculate_cronbach_alpha(df):
    """Calculates Cronbach's alpha for a dataframe of items."""
    k = df.shape[1]
    if k < 2:
        return np.nan
        
    item_variances = df.var(ddof=1)
    total_variance = df.sum(axis=1).var(ddof=1)
    
    if total_variance == 0:
        return 0.0
        
    alpha = (k / (k - 1)) * (1 - (item_variances.sum() / total_variance))
    return alpha

# Pre post trust comparison


In [10]:
# ...existing code...
def compare_pre_post(df, question_pairs, likert_scale, section="explanation_trust"):
    pairs = question_pairs[section]

    rows = []
    per_person = pd.DataFrame(index=df.index)

    for item, cols in pairs.items():
        pre = df[cols["pre"]].map(likert_scale)
        post = df[cols["post"]].map(likert_scale)
        delta = post - pre

        per_person[f"{item}_pre"] = pre
        per_person[f"{item}_post"] = post
        per_person[f"{item}_delta"] = delta

        rows.append({
            "item": item,
            "n_paired": int((pre.notna() & post.notna()).sum()),
            "pre_mean": pre.mean(),
            "post_mean": post.mean(),
            "delta_mean": delta.mean(),
        })

    item_summary = pd.DataFrame(rows).set_index("item")

    pre_cols = [c for c in per_person.columns if c.endswith("_pre")]
    post_cols = [c for c in per_person.columns if c.endswith("_post")]
    delta_cols = [c for c in per_person.columns if c.endswith("_delta") and c != "overall_delta"]

    
    per_person["overall_pre"] = per_person[pre_cols].mean(axis=1, skipna=True)
    per_person["overall_post"] = per_person[post_cols].mean(axis=1, skipna=True)
    per_person["overall_delta"] = per_person["overall_post"] - per_person["overall_pre"]

    overall_summary = pd.Series({
        "n_people_with_any_pair": int(per_person["overall_delta"].notna().sum()),
        "overall_pre_mean": per_person["overall_pre"].mean(),
        "overall_post_mean": per_person["overall_post"].mean(),
        "overall_delta_mean": per_person["overall_delta"].mean(),
        "cronbach_alpha_pre": calculate_cronbach_alpha(per_person[pre_cols]),
        "cronbach_alpha_post": calculate_cronbach_alpha(per_person[post_cols]),
        "cronbach_alpha_delta": calculate_cronbach_alpha(per_person[delta_cols]),
    })

    return item_summary, overall_summary, per_person




In [11]:
from IPython.display import display, Markdown


all_results = {}

for group_name, gdf in groups.items():
    item_summary, overall_summary, scores = compare_pre_post(
        gdf, question_pairs, likert_scale, section="explanation_trust"
    )
    all_results[group_name] = {
        "item_summary": item_summary,
        "overall_summary": overall_summary,
        "scores": scores,
    }

    display(Markdown(f"### {group_name} (n={len(gdf)})"))
    display(item_summary)
    display(overall_summary)

### All Participants (n=15)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,15,0.333333,0.733333,0.400000
trust_item_2,15,0.600000,0.466667,-0.133333
trust_item_3,15,0.200000,0.466667,0.266667
trust_item_4,15,0.466667,0.533333,0.066667


n_people_with_any_pair    15.000000
overall_pre_mean           0.400000
overall_post_mean          0.550000
overall_delta_mean         0.150000
cronbach_alpha_pre         0.689414
cronbach_alpha_post        0.625235
cronbach_alpha_delta       0.275862
dtype: float64

### AI Domain Experts (n=2)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,2,0.0,1.5,1.5
trust_item_2,2,1.0,1.5,0.5
trust_item_3,2,-0.5,0.5,1.0
trust_item_4,2,0.0,0.5,0.5


...

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,4,1.00,0.75,-0.25
trust_item_2,4,0.25,0.50,0.25
trust_item_3,4,1.00,0.50,-0.50
trust_item_4,4,0.50,0.50,0.00


n_people_with_any_pair    4.000000
overall_pre_mean          0.687500
overall_post_mean         0.562500
overall_delta_mean       -0.125000
cronbach_alpha_pre       -0.842105
cronbach_alpha_post       0.592593
cronbach_alpha_delta     -0.095238
dtype: float64

### Aviation Layman (n=11)

,n_paired,pre_mean,post_mean,delta_mean
item,,,,
trust_item_1,11,0.090909,0.727273,0.636364
trust_item_2,11,0.727273,0.454545,-0.272727
trust_item_3,11,-0.090909,0.454545,0.545455
trust_item_4,11,0.454545,0.545455,0.090909


n_people_with_any_pair    11.000000
overall_pre_mean           0.295455
overall_post_mean          0.545455
overall_delta_mean         0.250000
cronbach_alpha_pre         0.816768
cronbach_alpha_post        0.637076
cronbach_alpha_delta       0.496392
dtype: float64

Explanation Satisfaction comparison

In [12]:
questions ={
    "object_based_saliency": "Q00027",
    "action_heatmap": "Q00028",
    "moe_explanation": "Q00029"   
}

subquestions = [f"SQ{i:03d}" for i in range(1, 7)] 
print(subquestions)

['SQ001', 'SQ002', 'SQ003', 'SQ004', 'SQ005', 'SQ006']


# Satisfaction Comparison

In [13]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown



def analyze_explanation_satisfaction(df, questions, subquestions, likert_scale):
    results = {}
    for q_name, q_code in questions.items():
        satisfaction_cols = [f"{q_code}[{sq}]" for sq in subquestions]
        satisfaction_data = df[satisfaction_cols].apply(lambda col: col.map(likert_scale.get))
        
        # Calculate Cronbach's alpha BEFORE adding the mean column
        alpha = calculate_cronbach_alpha(satisfaction_data[satisfaction_cols])
        
        # Calculate mean across subquestions for each respondent
        satisfaction_data[f"{q_name}_mean"] = satisfaction_data.mean(axis=1, skipna=True)
        
        # Calculate correlation between subquestions and mean
        correlations = {}
        for col in satisfaction_cols:
            corr = satisfaction_data[col].corr(satisfaction_data[f"{q_name}_mean"])
            correlations[col] = corr
        
        results[q_name] = {
            "item_data": satisfaction_data,
            "overall_mean": satisfaction_data[f"{q_name}_mean"].mean(),
            "overall_median": satisfaction_data[f"{q_name}_mean"].median(),
            "n_responses": int(satisfaction_data[f"{q_name}_mean"].notna().sum()),
            "cronbach_alpha": alpha,
            "correlations": correlations,
        }
    
    return results

# Run and display results
# (Assuming df, questions, subquestions, and likert_scale are defined above)
satisfaction_results = analyze_explanation_satisfaction(df, questions, subquestions, likert_scale)

for q_name, q_results in satisfaction_results.items():
    display(Markdown(f"### {q_name}"))
    display(q_results["item_data"])
    
    summary_metrics = {
        "overall_mean": q_results["overall_mean"],
        "overall_median": q_results["overall_median"],
        "n_responses": q_results["n_responses"],
        "cronbach_alpha": q_results["cronbach_alpha"]
    }
    display(pd.Series(summary_metrics))
    
    display(Markdown("Correlations with mean:"))
    display(pd.Series(q_results["correlations"]))

### object_based_saliency

,Q00027[SQ001],Q00027[SQ002],Q00027[SQ003],Q00027[SQ004],Q00027[SQ005],Q00027[SQ006],object_based_saliency_mean
id,,,,,,,
2,1,1,1,1,1,1,1.000000
4,0,1,-1,0,1,-1,0.000000
6,1,0,0,-1,0,1,0.166667
7,0,0,0,0,0,0,0.000000
8,1,1,2,2,-2,1,0.833333
12,2,0,-1,-2,-1,1,-0.166667
20,2,0,1,0,0,2,0.833333
21,1,-1,-1,-1,-1,-1,-0.666667
24,1,-1,-2,-2,0,-1,-0.833333


overall_mean       0.588889
overall_median     0.833333
n_responses       15.000000
cronbach_alpha     0.852167
dtype: float64

Correlations with mean:

Q00027[SQ001]    0.497398
Q00027[SQ002]    0.827506
Q00027[SQ003]    0.895246
Q00027[SQ004]    0.824582
Q00027[SQ005]    0.604489
Q00027[SQ006]    0.848150
dtype: float64

...

### moe_explanation

,Q00029[SQ001],Q00029[SQ002],Q00029[SQ003],Q00029[SQ004],Q00029[SQ005],Q00029[SQ006],moe_explanation_mean
id,,,,,,,
2,1,1,1,1,1,1,1.000000
4,0,-1,-1,-1,0,0,-0.500000
6,-2,-2,-2,-2,-2,-2,-2.000000
7,0,0,0,0,0,0,0.000000
8,1,1,1,1,1,-1,0.666667
12,-1,-2,1,1,-1,0,-0.333333
20,2,2,1,2,1,1,1.500000
21,2,1,0,1,1,0,0.833333
24,1,0,-2,0,-1,-1,-0.500000


overall_mean       0.422222
overall_median     0.666667
n_responses       15.000000
cronbach_alpha     0.956943
dtype: float64

Correlations with mean:

Q00029[SQ001]    0.896664
Q00029[SQ002]    0.930286
Q00029[SQ003]    0.901091
Q00029[SQ004]    0.887331
Q00029[SQ005]    0.956025
Q00029[SQ006]    0.886050
dtype: float64

Write that the alpha is generall high enough to justify the mean as a measure central tendency. Also interestingly for some explanations the alpha is higher.

# pre post mental model questions

Starting with the post mental model question since this can be easier interpreted since there is a clear right answer. The pre mental model question is more difficult to interpret since there is no clear right answer and the question is more about the confidence of the participants in their mental model. Also comparing pre with post mental models are tricky for some questions.

In [14]:
question_pairs["mental_models"]

{'saliency_safe_state': {'pre': 'Q00012', 'post': 'Q00022'},
 'saliency_conflict_state': {'pre': 'Q00013', 'post': 'Q00021'},
 'moe': {'pre': 'Q00015', 'post': 'Q00024'},
 'action_heatmaps': {'pre': 'Q00016', 'post': 'Q00025'}}

## post correctness of answers

### Saliency safe state

In [27]:
# for the saliency safe state the correct answer for all subquestions SQ001 to S005 is No influence

def check_saliency_safe_state(df, question_code, subquestions):
    correct_answer = "No influence"
    results = {}
    
    for sq in subquestions:
        col_name = f"{question_code}[{sq}]"
        if col_name in df.columns:
            counts = df[col_name].value_counts(dropna=False)
            total_responses = counts.sum()
            correct_count = counts.get(correct_answer, 0)
            accuracy = correct_count / total_responses if total_responses > 0 else np.nan
            
            results[sq] = {
                "total_responses": total_responses,
                "correct_count": correct_count,
                "accuracy": accuracy
            }
        else:
            results[sq] = {
                "total_responses": 0,
                "correct_count": 0,
                "accuracy": np.nan
            }
    
    return pd.DataFrame(results).T

subquestions = [f"SQ{i:03d}" for i in range(1, 6)]
saliency_safe_state_results = check_saliency_safe_state(df, question_pairs["mental_models"]["saliency_safe_state"]["post"], subquestions)
display(Markdown("### Saliency Safe State - Accuracy of 'No influence' Response"))
display(saliency_safe_state_results)

### Saliency Safe State - Accuracy of 'No influence' Response

,total_responses,correct_count,accuracy
SQ001,15.0,9.0,0.600000
SQ002,15.0,12.0,0.800000
SQ003,15.0,11.0,0.733333
SQ004,15.0,15.0,1.000000
SQ005,15.0,14.0,0.933333


### Saliency conflicting state

In [29]:
def check_saliency_conflict_state(df, question_code, subquestions):
    correct_answer_pairs = {
        "SQ001": ["Strong influence to turn left"], # Aircraft 1
        "SQ002": ["Weak influence to turn left","Strong influence to turn left"], #Aircraft 2
        "SQ003": ["No influence"], # Aircraft 5
    }
    results = {}
    
    for sq in subquestions:
        col_name = f"{question_code}[{sq}]"
        if col_name in df.columns:
            counts = df[col_name].value_counts(dropna=False)
            total_responses = counts.sum()
            # Get correct answers for this subquestion
            correct_answers = correct_answer_pairs.get(sq, [])
            # Sum counts for all correct answers
            correct_count = sum(counts.get(ans, 0) for ans in correct_answers)
            accuracy = correct_count / total_responses if total_responses > 0 else np.nan
            
            results[sq] = {
                "total_responses": total_responses,
                "correct_count": correct_count,
                "accuracy": accuracy
            }
        else:
            results[sq] = {
                "total_responses": 0,
                "correct_count": 0,
                "accuracy": np.nan
            }
    
    return pd.DataFrame(results).T

subquestions_conflict = [f"SQ{i:03d}" for i in range(1, 4)]
saliency_conflict_state_results = check_saliency_conflict_state(df, question_pairs["mental_models"]["saliency_conflict_state"]["post"], subquestions_conflict)
display(Markdown("### Saliency Conflict State - Accuracy of Correct Responses"))

display(saliency_conflict_state_results)

### Saliency Conflict State - Accuracy of Correct Responses

,total_responses,correct_count,accuracy
SQ001,15.0,12.0,0.8
SQ002,15.0,9.0,0.6
SQ003,15.0,15.0,1.0


In [32]:
#print(df["Q00021[SQ001]"])

### Moe post

In [56]:
# Subquestions SQ001 to S003 
# correct answers are: SQ001 Controlling , SQ002 Evading, SQ003 Both equally

def check_moe_post_correct(df,question_code, subquestions):
    correct_answer_pairs = {
        "SQ001": ["Controlling"], 
        "SQ002": ["Evading"],
        "SQ003": ["Both equally"],
    }
    results = {}
    
    for sq in subquestions:
        col_name = f"{question_code}[{sq}]"
        if col_name in df.columns:
            counts = df[col_name].value_counts(dropna=False)
            total_responses = counts.sum()
            # Get correct answers for this subquestion
            correct_answers = correct_answer_pairs.get(sq, [])
            # Sum counts for all correct answers
            correct_count = sum(counts.get(ans, 0) for ans in correct_answers)
            accuracy = correct_count / total_responses if total_responses > 0 else np.nan
            
            results[sq] = {
                "total_responses": total_responses,
                "correct_count": correct_count,
                "accuracy": accuracy,
            }
        else:
            results[sq] = {
                "total_responses": 0,
                "correct_count": 0,
                "accuracy": np.nan
            }
    
    return pd.DataFrame(results).T

moe_post_results = check_moe_post_correct(df, question_pairs["mental_models"]["moe"]["post"], ["SQ001", "SQ002", "SQ003"])
display(Markdown("### MOE Post - Accuracy of Correct Responses"))
display(moe_post_results)


### MOE Post - Accuracy of Correct Responses

,total_responses,correct_count,accuracy
SQ001,15.0,13.0,0.866667
SQ002,15.0,13.0,0.866667
SQ003,15.0,10.0,0.666667


In [55]:
print(df[question_pairs["mental_models"]["moe"]["post"]+"[SQ003]"])

id
2      Both equally
4       Controlling
6     I cannot tell
7     I cannot tell
8      Both equally
12     Both equally
20      Controlling
21     Both equally
24     Both equally
26     Both equally
27      Controlling
34     Both equally
35     Both equally
36     Both equally
38     Both equally
Name: Q00024[SQ003], dtype: str


### Safe state vs background agreement

In [45]:
# question code == Q00023 SQ001 (safe state) and SQ002 (background)
# used likert_scale there for mapping can be from -2 to 2 from strongly disagree to strongly agree
# for each subquestion this score will be calculated by its own in the end we get 2 agreement scores for each SQ
def check_safe_state_background_agreement(df, question_code, subquestions, likert_scale):
    results = {}

    for sq in subquestions:
        col_name = f"{question_code}[{sq}]"
        if col_name in df.columns:
            mapped_responses = df[col_name].map(likert_scale)
            results[sq] = {
                "n_responses": int(mapped_responses.notna().sum()),
                "mean_agreement": mapped_responses.mean(),
                "variance_agreement": mapped_responses.var()
            }
        else:
            results[sq] = {
                "n_responses": 0,
                "mean_agreement": np.nan
            }

    return pd.DataFrame(results).T


# Q00023 uses: Strongly Disagree, Disagree, Neutral, Agree, Strongly Agree
likert_scale2 = {
    "Strongly Disagree": -2,
    "Disagree": -1,
    "Neutral": 0,
    "Agree": 1,
    "Strongly Agree": 2,
}

subquestions_agreement = [f"SQ{i:03d}" for i in range(1, 3)]
agreement_results = check_safe_state_background_agreement(
    df, "Q00023", subquestions_agreement, likert_scale2
)

display(Markdown("### Safe State vs Background Agreement"))
display(agreement_results)


### Safe State vs Background Agreement

,n_responses,mean_agreement,variance_agreement
SQ001,15.0,0.533333,1.695238
SQ002,15.0,1.266667,0.780952


Interestingly for the safe state shap SQ001 the mean agreement is lower but thats because a minority pulled it down. Even maybe stonger than backgroudn shap but because some voted disagree its lower in mean.

In [48]:
print(df["Q00023[SQ001]"])

id
2                 Agree
4              Disagree
6     Strongly Disagree
7               Neutral
8        Strongly Agree
12                Agree
20             Disagree
21             Disagree
24       Strongly Agree
26       Strongly Agree
27                Agree
34       Strongly Agree
35                Agree
36              Neutral
38                Agree
Name: Q00023[SQ001], dtype: str


### action heatmap

In [58]:
# correct answerers are Strong left and Weak left there is no subquestion for this one

def check_action_heatmap_correct(df,question_code):
    correct_answers = ["Strong left", "Weak left"]
    col_name = f"{question_code}"
    if col_name in df.columns:
        counts = df[col_name].value_counts(dropna=False)
        total_responses = counts.sum()
        correct_count = sum(counts.get(ans, 0) for ans in correct_answers)
        accuracy = correct_count / total_responses if total_responses > 0 else np.nan
        
        results = {
            "total_responses": total_responses,
            "correct_count": correct_count,
            "accuracy": accuracy,
        }
    else:
        results = {
            "total_responses": 0,
            "correct_count": 0,
            "accuracy": np.nan
        }
    
    return pd.Series(results)

action_heatmap_results = check_action_heatmap_correct(df, question_pairs["mental_models"]["action_heatmaps"]["post"])
display(Markdown("### Action Heatmap - Accuracy of Correct Responses"))
display(action_heatmap_results)

### Action Heatmap - Accuracy of Correct Responses

total_responses    15.0
correct_count       9.0
accuracy            0.6
dtype: float64

In [59]:
print(df[question_pairs["mental_models"]["action_heatmaps"]["post"]])

id
2        No action
4      Cannot tell
6        Weak left
7        No action
8      Cannot tell
12     Strong left
20     Cannot tell
21    Strong right
24       Weak left
26       Weak left
27     Strong left
34     Strong left
35     Strong left
36     Strong left
38     Strong left
Name: Q00025, dtype: str


## pre correctness of answers